In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

1. Model Parallelism

In [ ]:
class ModelParallelLLM(nn.Module):
    def __init__(self, vocab_size, num_layers=8, num_gpus=4):
        super().__init__()
        self.num_gpus = num_gpus
        
        layers_per_gpu = num_layers // num_gpus
        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.TransformerEncoderLayer(
                    d_model=512,
                    nhead=8,
                    dim_feedforward=2048,
                    dropout=0.1,
                    batch_first=True
                ) for _ in range(layers_per_gpu)
            ]).to(f'cuda:{i}') for i in range(num_gpus)
        ])
    
    def forward(self, x):
        for gpu_layers in self.layers:
            for layer in gpu_layers:
                x = layer(x.to(layer.weight.device))
        return x

2. Gradient Checkpointing (Memory Efficiency)

In [2]:
from torch.utils.checkpoint import checkpoint

class MemoryEfficientLLM(nn.Module):
    def forward(self, x):
        for layer in self.layers:
            x = checkpoint(layer, x)
        return x

3. Mixed Precision Training

In [4]:
from torch.amp import autocast, GradScaler

In [ ]:
scaler = GradScaler()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for batch in train_loader:
    optimizer.zero_grad()
    
    with autocast():
        logits = model(input_ids)
        loss = criterion(logits, labels)
    
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

4. Data Parallelism

In [ ]:
model = nn.DataParallel(model, device_ids=[0, 1, 2, 3])

**Distributed Training**

In [ ]:
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import os

In [ ]:
def setup_distributed():
    dist.init_process_group(backend='nccl')
    torch.cuda.set_device(int(os.environ['LOCAL_RANK']))

def train_distributed(model, train_loader):
    model = DDP(model, device_ids=[int(os.environ['LOCAL_RANK'])])
    
    for batch in train_loader:
        loss = model(batch)
        loss.backward()
        optimizer.step()